# RouterBench v2 Balanced ModernBERT 전체 학습 — Colab 중단 재개

이 notebook은 Colab의 임시 GPU 디스크에서 학습하고, 완료된 resume checkpoint만 Google Drive에 복제합니다. 세션이 끊기면 새 GPU 세션에서 모든 셀을 다시 실행하세요. `modernbert-colab-train --resume-mode auto`가 Drive의 최신 정상 checkpoint를 찾아 이어서 학습합니다.

학습 수학은 공통 `modernbert_training.py`를 사용하고, Drive 동기화와 자동 재개만 Colab 전용 코드가 담당합니다. v2 train은 easy/medium/hard 각 6,256건이며, micro-batch 1 × gradient accumulation 8의 각 optimizer 단위에 세 클래스가 3/3/2로 들어가고 2건을 더 받는 클래스는 순환합니다. RouterBench validation은 각 epoch의 약 33%·66%·100%에서 측정해 총 9개 진행 지점을 남깁니다. 각 지점의 stdout과 Drive `loss_progress.svg`에는 직전 구간 train loss와 validation loss가 함께 기록됩니다. 각 epoch 종료 후 RouterArena 전체 데이터는 **추론 관찰에만** 사용됩니다. 두 평가 결과 모두 gradient, optimizer, early stopping, checkpoint 선택에 사용되지 않습니다.

In [ ]:
import platform
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print({"python": platform.python_version(), "gpu": gpu.stdout.strip() or None})
if gpu.returncode != 0:
    raise RuntimeError("GPU runtime이 아닙니다. GPU가 있는 Colab server를 선택하세요.")

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE_RUN = Path("/content/drive/MyDrive/LLM_Router/training/modernbert-router-balanced-v2")
DRIVE_RUN.parent.mkdir(parents=True, exist_ok=True)
print("Persistent run:", DRIVE_RUN)

In [ ]:
from pathlib import Path
import json
import os
from pathlib import Path
import shutil
import tarfile

archive = Path("/content/modernbert-colab-balanced-v2-input.tar.gz")
workspace = Path("/content/LLM_Router")
if not archive.is_file():
    raise FileNotFoundError(
        "modernbert-colab-balanced-v2-input.tar.gz를 Colab /content에 먼저 업로드하세요."
    )
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True)
with tarfile.open(archive, "r:gz") as bundle:
    bundle.extractall(workspace, filter="data")
print("Extracted:", workspace)

In [ ]:
import shutil
import subprocess
import sys

def install(label, *arguments):
    print(f"\n[install] {label}", flush=True)
    command = [
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "--no-cache-dir",
        "--break-system-packages",
        *arguments,
    ]
    print(" ".join(command), flush=True)
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout, flush=True)
    if result.returncode != 0:
        output_tail = "\n".join(result.stdout.splitlines()[-40:])
        raise RuntimeError(
            f"{label} installation failed with exit code {result.returncode}.\n"
            f"Last pip output:\n{output_tail}"
        )
    return result.stdout

def uninstall_optional(*packages):
    command = [
        sys.executable, "-m", "pip", "uninstall",
        "--yes", "--break-system-packages", *packages,
    ]
    print("\n[uninstall incompatible optional packages]", flush=True)
    print(" ".join(command), flush=True)
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout, flush=True)
    if result.returncode != 0:
        raise RuntimeError("Optional package removal failed:\n" + result.stdout)

disk = shutil.disk_usage("/content")
print({
    "python": sys.version,
    "executable": sys.executable,
    "content_free_gib": round(disk.free / 1024**3, 2),
})
PYTORCH_INDEX = "https:" + "//download.pytorch.org/whl/cu124"
PYPI_INDEX = "https:" + "//pypi.org/simple"
torch_install_output = install(
    "PyTorch 2.5.1 CUDA 12.4",
    "--index-url", PYTORCH_INDEX,
    "--extra-index-url", PYPI_INDEX,
    "--requirement", "/content/LLM_Router/ml/requirements-training-cu124.txt",
)
library_install_output = install(
    "pinned training libraries",
    "--requirement", "/content/LLM_Router/ml/requirements-training.lock",
)
uninstall_optional("torchvision", "torchaudio")
install(
    "LLM Router package",
    "--no-deps", "-e", "/content/LLM_Router/ml",
)
verify_code = """
import importlib.util
import json
import numpy
import pandas
import pyarrow
import pyarrow.parquet
import torch
import transformers
print(json.dumps({
    'torch': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'transformers': transformers.__version__,
    'numpy': numpy.__version__,
    'pandas': pandas.__version__,
    'pyarrow': pyarrow.__version__,
    'cuda_available': torch.cuda.is_available(),
    'torchvision_installed': importlib.util.find_spec('torchvision') is not None,
}, sort_keys=True))
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable')
if importlib.util.find_spec('torchvision') is not None:
    raise RuntimeError('Incompatible optional torchvision is still installed')
"""
verification = subprocess.run(
    [sys.executable, "-c", verify_code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print("[fresh-process verification]", verification.stdout, flush=True)
if verification.returncode != 0:
    raise RuntimeError("Fresh-process runtime verification failed:\n" + verification.stdout)
compiled_packages_changed = any(
    marker in (torch_install_output + library_install_output)
    for marker in ("Successfully installed", "Attempting uninstall")
)
restart_marker = Path("/content/.llm_router_kernel_restart_required")
if compiled_packages_changed:
    restart_marker.write_text(str(os.getpid()), encoding="utf-8")
    print("\nIMPORTANT: compiled packages changed. Restart the notebook kernel, then Run All.", flush=True)
else:
    print("Runtime packages were already current; no new kernel restart is required.", flush=True)

In [ ]:
import os
from pathlib import Path

restart_marker = Path("/content/.llm_router_kernel_restart_required")
if restart_marker.is_file():
    installed_in_pid = restart_marker.read_text(encoding="utf-8").strip()
    if installed_in_pid == str(os.getpid()):
        raise RuntimeError(
            "Compiled Python packages were replaced in this kernel. "
            "Restart the notebook kernel and then Run All before reading Parquet."
        )
    restart_marker.unlink()
import pandas as pd
import pyarrow

TRAINING_PATH = Path("/content/LLM_Router/ml/artifacts/routerbench-training/withmartian-routerbench-2024-03-27__minimum-sufficient-tier-v2__routerbench-training-preprocess-v2-balanced/training.parquet")
CONFIG_PATH = Path("/content/LLM_Router/ml/config/modernbert_t4_balanced_v2.json")
RUNTIME_PATH = Path("/content/LLM_Router/ml/config/modernbert_runtime_cu124_v1.json")
ROUTERARENA_PATH = Path("/content/LLM_Router/ml/artifacts/routerarena-evaluation-full/evaluation.parquet")
ROUTERARENA_METADATA_PATH = Path("/content/LLM_Router/ml/artifacts/routerarena-evaluation-full/metadata.parquet")
ROUTERARENA_CONFIG_PATH = Path("/content/LLM_Router/ml/config/routerarena_evaluation.json")
rows = pd.read_parquet(TRAINING_PATH, columns=["split", "difficulty_label"])
routerarena_rows = pd.read_parquet(ROUTERARENA_PATH, columns=["evaluation_split"])
print({
    "routerbench_rows": len(rows),
    "routerbench_splits": rows["split"].value_counts().to_dict(),
    "train_labels": rows.loc[rows["split"].eq("train"), "difficulty_label"].value_counts().to_dict(),
    "routerarena_inference_only_rows": len(routerarena_rows),
})

In [ ]:
import os
import shutil
import subprocess

environment = os.environ.copy()
environment["HF_HOME"] = "/content/huggingface-cache"
environment["PYTHONUNBUFFERED"] = "1"
executable = shutil.which("modernbert-colab-train")
if executable is None:
    raise RuntimeError("modernbert-colab-train이 없습니다. 설치 셀을 다시 실행하세요.")
command = [
    executable,
    "--config", str(CONFIG_PATH),
    "--runtime-contract", str(RUNTIME_PATH),
    "--training-path", str(TRAINING_PATH),
    "--local-run-dir", "/content/llm-router-training/modernbert-router-balanced-v2",
    "--persistent-run-dir", str(DRIVE_RUN),
    "--device", "cuda",
    "--resume-mode", "auto",
    "--save-every-optimizer-steps", "50",
    "--drive-sync-every-optimizer-steps", "500",
    "--keep-last-checkpoints", "2",
    "--routerbench-validations-per-epoch", "3",
    "--routerarena-evaluation-path", str(ROUTERARENA_PATH),
    "--routerarena-metadata-path", str(ROUTERARENA_METADATA_PATH),
    "--routerarena-config", str(ROUTERARENA_CONFIG_PATH),
    "--routerarena-device", "cuda",
    "--routerarena-batch-size", "1",
]
print("Starting or resuming training...")
print(" ".join(command), flush=True)
process = subprocess.Popen(
    command,
    env=environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
training_output = []
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
    training_output.append(line.rstrip())
    if len(training_output) > 200:
        training_output.pop(0)
return_code = process.wait()
if return_code != 0:
    output_tail = "\n".join(training_output[-100:])
    raise RuntimeError(
        f"ModernBERT training failed with exit code {return_code}.\n"
        f"Last training output:\n{output_tail}"
    )

In [ ]:
import json

checkpoints = sorted((DRIVE_RUN / "checkpoints").glob("step-*"))
final_model = DRIVE_RUN / "final-model"
print("Completed checkpoints:", [path.name for path in checkpoints])
print("Final model complete:", (final_model / "_SUCCESS").is_file())
if (final_model / "metrics.json").is_file():
    metrics = json.loads((final_model / "metrics.json").read_text())
    print({
        "optimizer_steps": metrics["optimization"]["optimizer_steps"],
        "resumed_from_step": metrics["optimization"]["resumed_from_step"],
        "validation_accuracy": metrics["validation"]["accuracy"],
        "validation_macro_f1": metrics["validation"]["macro_f1"],
        "reload_passed": metrics["checkpoint"]["reload_smoke_passed"],
    })

## RouterBench 학습 중 validation 변화 보기

각 epoch의 약 1/3·2/3·3/3 지점에서 고정된 RouterBench validation split을 추론한 결과입니다. 총 9개 점의 accuracy와 macro-F1은 높을수록, loss는 낮을수록 좋습니다. 이 값은 관찰용이며 학습 중단이나 best checkpoint 선택에 사용되지 않습니다.

In [ ]:
from IPython.display import SVG, display
import json
import pandas as pd

validation_root = DRIVE_RUN / "routerbench-validation-progress"
progress_path = validation_root / "validation_progress.json"
if not progress_path.is_file():
    raise FileNotFoundError("아직 완료된 RouterBench 중간 검증이 없습니다. 학습 셀을 실행하세요.")
validation_progress = json.loads(progress_path.read_text())
display(pd.DataFrame(validation_progress["observations"]))
display(SVG(filename=str(validation_root / "validation_progress.svg")))

## Epoch별 RouterArena 변화 보기

아래 표와 그래프는 epoch 1·2·3 모델을 같은 RouterArena 데이터로 추론한 관찰 결과입니다. Accuracy와 Macro F1은 높을수록, under-routing 비율은 낮을수록 좋습니다. 두 데이터셋의 라벨 생성 의미가 완전히 같지는 않으므로 절대 성능이 아니라 epoch 사이의 변화와 분포를 중심으로 해석합니다.

In [ ]:
from IPython.display import SVG, display
import json
import pandas as pd

epoch_root = DRIVE_RUN / "routerarena-epoch-evaluations"
progress_path = epoch_root / "epoch_progress.json"
if not progress_path.is_file():
    raise FileNotFoundError("아직 완료된 epoch 평가가 없습니다. 학습 셀을 실행하세요.")
progress = json.loads(progress_path.read_text())
display(pd.DataFrame(progress["epochs"]))
display(SVG(filename=str(epoch_root / "epoch_progress.svg")))
for epoch in progress["epochs"]:
    epoch_dir = epoch_root / f"epoch-{epoch['epoch_number']:02d}"
    print(f"Epoch {epoch['epoch_number']} predicted-group heatmap")
    display(SVG(filename=str(epoch_dir / "predicted_group_heatmap.svg")))

## 세션이 끊겼을 때

1. 새 Colab GPU server를 만듭니다.
2. 같은 입력 archive를 다시 업로드합니다.
3. 이 notebook의 셀을 처음부터 다시 실행합니다.
4. 같은 `DRIVE_RUN`, config, training Parquet을 유지합니다.

Drive 복사 도중 끊겨 생긴 `.partial-*` 디렉터리는 정상 checkpoint로 인식하지 않습니다. 이 v2 실험은 v1 checkpoint와 fingerprint가 다르므로 반드시 `modernbert-router-balanced-v2` 경로를 사용하세요.